## Install Dependencies

In [ ]:
# Install dependencies
%pip install -q amplpy matplotlib

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["gurobi", "xpress", "highs", "gokestrel"], # modules to install
    license_uuid="default") # license to use

### AMPL Formulation
#### Membership operator `in`, `not in`

In [2]:
%%ampl_eval
reset;                      # clear previous model/data

set PRODUCTS;               # all products
set SPECIAL;                # subset of special products

param is_special {p in PRODUCTS} :=
    if p in SPECIAL then 1 else 0;   # 1 if p is in SPECIAL

param is_regular {p in PRODUCTS} :=
    if p not in SPECIAL then 1 else 0;  # 1 if p is NOT in SPECIAL

data;
set PRODUCTS := A B C;      # define products
set SPECIAL := B C;         # define special subset

display is_special, is_regular;  # show results

: is_special is_regular    :=
A      0          1
B      1          0
C      1          0
;



#### Subset operator `within`

In [3]:
%%ampl_eval
reset;                         # clear previous model/data

set ALL_ITEMS;                 # universal set
set SUB1 within ALL_ITEMS;     # SUB1 must be a subset of ALL_ITEMS

data;
set ALL_ITEMS = A, B, C, D;    # define main set
set SUB1 = A, C;               # valid subset (only uses elements from ALL_ITEMS)

display SUB1;                  # output SUB1

set SUB1 := A C;



#### Cardinality function

In [4]:
%%ampl_eval
reset;

set PRODUCTS;                     # all available items
set SELECTED within PRODUCTS;     # chosen items (must be subset of PRODUCTS)

param num_products := card(PRODUCTS);   # total number of products

# Limit how many items can be selected
subject to limit_selection:
    card(SELECTED) <= 2;          # at most 2 items can be selected

data;

set PRODUCTS = A, B, C, D;        # define all products
set SELECTED = A, C;              # choose two products

display num_products, card(SELECTED);  # show total and selected count

num_products = 4
card(SELECTED) = 2



#### Set-of operator (setof)

In [5]:
%%ampl_eval
reset;  # Clear previous model data

# Simple projection
model;              # The 'model;' statement marks the model formulation section.
set PAIRS dimen 2;  # Define 2D set of tuples
set FIRST_COMPONENTS :=
    setof {(i,j) in PAIRS} (i);  # Extract first elements from PAIRS

data;                           # The 'data;' statement marks the beginning of data input section.
set PAIRS := (A,1) (A,2) (B,1);  # Sample pair data


# Unique values
model;              # The 'model;' statement marks the model formulation section.
param T = 5;  # Number of cost entries
param cost{1..T} >= 0;  # Non-negative cost array
set UNIQUE_COSTS :=
    setof {i in 1..T} cost[i];  # Get distinct cost values (10,20,30)

data;
param cost:= 1 10 2 20 3 10 4 30 5 20;  # Costs with duplicates


# conditional set construction
model;
set PRODUCTS;  # Product names set
param profit {PRODUCTS};  # Profit per product
set PROFITABLE :=
    setof {p in PRODUCTS: profit[p] > 0} (p);  # Filter products with positive profit

data;
set PRODUCTS := A B C;
param profit := 
A 5
B -2
C 3;  # B is unprofitable


display FIRST_COMPONENTS, UNIQUE_COSTS, PROFITABLE;   # {A, B}, {10,20,30}, {A,C}

set FIRST_COMPONENTS := A B;

set UNIQUE_COSTS := 10 20 30;

set PROFITABLE := A C;

